# Gemini によるセーフガード

## 概要

大規模言語モデル（LLM）は、言語の翻訳、テキストの要約、クリエイティブな文章の作成、コードの生成、チャットボットやバーチャルアシスタントの強化、検索エンジンや推薦システムの補完などを行うことができます。LLM の驚くべき汎用性は、意図しない、あるいは予期しないどのような種類の出力が生成されるかを正確に予測することを難しくする要因でもあります。 

こうしたリスクと複雑さを考慮し、Gemini は [Google の AI 原則](https://ai.google/responsibility/principles/) を念頭に置いて設計されています。しかし、開発者が安全かつ責任を持ってモデルをデプロイするには、モデルを理解してテストすることが重要です。開発者を支援するために、Agent Platform Studio にはコンテンツフィルタリング、安全性評価（safety ratings）、およびユースケースやビジネスに適した安全性フィルタのしきい値を定義する機能が組み込まれています。

詳細については、[責任ある AI に関する Google Cloud 生成 AI ドキュメント](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/responsible-ai) を参照してください。

## 学習目標

このノートブックでは、Python SDK を使用して Gemini から返される安全性評価（safety ratings）を検査する方法と、Gemini からのレスポンスをフィルタリングするための安全性しきい値を設定する方法を学びます。

実行する手順は以下のとおりです。

- Gen AI SDK 経由で Gemini を呼び出し、レスポンスの安全性評価を検査する
- ニーズに応じて安全性評価をフィルタリングするためのしきい値を定義する

## はじめに

### Google Cloud の定義

In [ ]:
PROJECT_ID = !gcloud config get-value project  # noqa: E999
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"

### ライブラリのインポート

In [ ]:
from google import genai
from google.genai.types import (
    GenerateContentConfig,
    HarmBlockThreshold,
    HarmCategory,
    Part,
    SafetySetting,
)

### Gemini 用 GenerateContentConfig のセットアップ

In [ ]:
MODEL = "gemini-3.8-flash"
client = genai.Client(vertexai=True, location="global")

# レスポンスのばらつきを抑えるためのパラメータを設定
generation_config = GenerateContentConfig(
    safety_settings=[
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=HarmBlockThreshold.BLOCK_NONE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=HarmBlockThreshold.BLOCK_NONE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=HarmBlockThreshold.BLOCK_NONE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            threshold=HarmBlockThreshold.BLOCK_NONE,
        ),
    ]
)

## テキストの生成と安全性評価の表示

まずは、Gemini を使用して心地よい響きのテキストレスポンスを生成することから始めましょう。

In [ ]:
# Gemini を呼び出す
nice_prompt = "Say three nice things about me"
responses = client.models.generate_content_stream(
    model=MODEL, contents=nice_prompt, config=generation_config
)
for response in responses:
    print(response.text, end="")

#### 安全性評価の検査

ストリーミングレスポンスの `safety_ratings` を確認します。

In [ ]:
response.candidates[0].to_json_dict()

#### 安全性評価の理解: カテゴリと確率

各 `category`（カテゴリ）タイプとそれに関連付けられた `probability`（確率）ラベルを含む安全性評価を確認できます。

`category` タイプには以下が含まれます。

* ヘイトスピーチ: `HARM_CATEGORY_HATE_SPEECH`
* 危険なコンテンツ: `HARM_CATEGORY_DANGEROUS_CONTENT`
* ハラスメント: `HARM_CATEGORY_HARASSMENT`
* 露骨な性的表現: `HARM_CATEGORY_SEXUALLY_EXPLICIT`

`probability` ラベルは以下のとおりです。

* `NEGLIGIBLE` - コンテンツが安全でない可能性はごくわずかです
* `LOW` - コンテンツが安全でない可能性は低いです
* `MEDIUM` - コンテンツが安全でない可能性は中程度です
* `HIGH` - コンテンツが安全でない可能性は高いです

`probability_score` は、各安全性カテゴリに関する [0, 1] の確率スコアを意味します。<br>
ここでは非常に低い値が表示されているはずです。

これらのカテゴリのいずれかをトリガーする可能性のあるプロンプトを試してみましょう:

In [ ]:
impolite_prompt = "Write a list of 5 disrespectful things that I might say to the universe after stubbing my toe in the dark:"

response = client.models.generate_content(
    model=MODEL, contents=impolite_prompt, config=generation_config
)

response.candidates[0].to_json_dict()

Gemini 自体が有害である可能性のあるプロンプトを適切に処理するため、より高い確率のカテゴリは表示されないかもしれませんが、`probability_score` が前のプロンプトよりも高くなっていることが観察できる場合があります。

### 安全性評価のしきい値の定義

ビジネスのポリシーやユースケースに応じて、デフォルトの安全性フィルタのしきい値を調整したい場合があります。Gemini では、各カテゴリにしきい値を渡す方法が提供されています。

以下のリストは、設定可能なブロックしきい値ラベルを示しています。

* `BLOCK_ONLY_HIGH` - 安全でないコンテンツの可能性が高いと検出された場合にブロック
* `BLOCK_MEDIUM_AND_ABOVE` - 安全でないコンテンツの可能性が中程度または高いと検出された場合にブロック
* `BLOCK_LOW_AND_ABOVE` - 安全でないコンテンツの可能性が低い、中程度、または高いと検出された場合にブロック
* `BLOCK_NONE` - 安全でないコンテンツの可能性に関係なく、常に表示

#### 安全性しきい値の設定
以下では、安全性しきい値が最も感度の高いしきい値である `BLOCK_LOW_AND_ABOVE` に設定されています。

In [ ]:
generation_config = GenerateContentConfig(
    safety_settings=[
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
    ]
)

#### しきい値のテスト

ここでは、先ほどの失礼なプロンプトを、最も感度の高い安全性しきい値とともに再利用します。`LOW` の確率ラベルであっても、レスポンスがブロックされるはずです。

ブロックされたレスポンスが表示されるまで、複数回試してみてください。

In [ ]:
impolite_prompt = "Write a list of 5 disrespectful things that I might say to the universe after stubbing my toe in the dark:"

response = client.models.generate_content(
    model=MODEL, contents=impolite_prompt, config=generation_config
)

response.candidates[0].to_json_dict()

このノートブックは [Thu Ya Kyaw](https://github.com/iamthuya) 氏の成果に基づいています。<br>
https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/responsible-ai/gemini_safety_ratings.ipynb

Copyright 2024 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License